# Train **llmscratch** on Google Colab

Trains the from-scratch LLM on a Colab GPU (**PyTorch/CUDA track**). Auto-enables the right precision: **bf16** on A100/L4, **fp16** on a free T4 (Turing has no bf16 tensor cores), plus `torch.compile` + TF32 + fused AdamW.

**First:** `Runtime -> Change runtime type -> GPU`.

Runtimes are ephemeral, so checkpoints go to Drive and we use `--resume`. Data lives on **local disk** (Drive is slow for the random reads training does).

In [ ]:
!nvidia-smi   # T4 = free; A100/L4 = Colab Pro

## 1. Get the code + install

In [ ]:
!git clone https://github.com/wishbone305/llm_scratch.git
%cd llm_scratch
!pip install -q -e .        # torch preinstalled; mlx auto-skips
!pip install -q datasets    # for streaming datasets (Options B/C)

## 2. Checkpoints -> Drive, data -> local disk
Checkpoints persist on Drive; the `.bin` data is copied to Colab's **local disk** so batch reads are fast (reading `.bin` off mounted Drive is a major slowdown).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/llmscratch'
os.makedirs(f'{DRIVE}/data', exist_ok=True); os.makedirs(f'{DRIVE}/out', exist_ok=True)
OUT = f'{DRIVE}/out'                 # checkpoints -> Drive (survive disconnects)
os.makedirs('data', exist_ok=True)   # data -> LOCAL disk (fast)
for f in ('train.bin', 'val.bin'):
    src = f'{DRIVE}/data/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'data/{f}'); print('copied', f, 'to local disk')
!ls -la data/

## 3. Get the data — pick ONE (skip if bins were copied above)
**A:** upload `train.bin`/`val.bin` to `MyDrive/llmscratch/data/` once (auto-copied above).
**B:** stream FineWeb-Edu (~1B tokens).
**C:** build the **blended ~2-4B-token corpus** (best — fixes the data limit).

In [ ]:
# Option B: FineWeb-Edu only
from datasets import load_dataset
from llmscratch.data import write_documents
import itertools
it = iter(load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT', split='train', streaming=True))
write_documents((e['text'] for e in itertools.islice(it, 500_000)), 'data/train.bin')
write_documents((e['text'] for e in itertools.islice(it, 4_000)),   'data/val.bin')
!cp data/*.bin {DRIVE}/data/    # save to Drive for next session

In [ ]:
# Option C: blended corpus (recommended) — edit target for your GPU/time budget
!python scripts/build_mix.py --target-tokens 2e9
!cp data/*.bin {DRIVE}/data/

## 4. Train
- **Free T4:** `colab_t4` (big batch + gradient checkpointing + fp16 — sized for 16 GB).
- **A100 / L4 (Pro):** `gpt_200m` — the full ~205M target.

`--resume` continues from Drive after any disconnect (just re-run this cell).

In [ ]:
CONFIG = 'colab_t4'    # A100/L4 -> 'gpt_200m'
!python -m llmscratch.train_torch --config {CONFIG} --data-dir data --out-dir {OUT} --resume

## 5. Watch training

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Sample from a checkpoint

In [ ]:
!python -m llmscratch.sample_torch --ckpt {OUT}/{CONFIG}.pt \
    --prompt 'The Sun is a star that' --max-new-tokens 150

---
### Reconnect / speed tips
- **Disconnect?** Re-run cells **1, 2, 4** (bins are on Drive -> copied local); `--resume` picks up.
- **Slow?** Confirm `nvidia-smi` shows a GPU, keep the tab active, and make sure data is on local `data/` (cell 2), not read from Drive. A free T4 is an old GPU — Colab Pro's A100/L4 is ~10-30x faster and the right home for `gpt_200m` on a multi-billion-token blend.